# Đo BM25 + cross-encoder rerank — UIT DSC 2026 Task 2 v10

**Đây là lần chạy ĐO, không phải bài nộp.** Cần Internet ON để clone repo và tải reranker.

Cần gắn trước khi chạy:

| loại | thứ cần gắn | dùng làm gì |
|---|---|---|
| Accelerator | **GPU T4 x2** | chạy XLM-R large 568M |
| Internet | **ON** | clone GitHub + tải HuggingFace |
| Input | dữ liệu thi — thư mục chứa `selected-contexts/` và `train.json` | dựng corpus/chunk/gold |

Model KHÔNG cần gắn tay: cell 4 tải `AITeamVN/Vietnamese_Reranker` từ HuggingFace.
Nếu bạn đã upload nó thành Kaggle Dataset/Model thì gắn vào, cell 4 tự nhận.

Không upload `chunks.jsonl`/`bm25.pkl` (1,2 GB): notebook dựng lại từ dữ liệu thi, ~12 phút CPU.

## 1 — Nạp codebase từ Dataset

Codebase vào bằng **Kaggle Dataset** `legalqa-v10-src`, không clone GitHub: repo private,
fine-grained PAT qua Kaggle Secrets tốn hai vòng debug mà vẫn 404.

Cần gắn: **Input → Add Input → Datasets → `legalqa-v10-src`**

Dòng `codebase:` in ra commit của bản đang chạy. Đối chiếu với `git log -1` trên máy dev —
lệch nghĩa là dataset cũ, chạy `./scripts/push_dataset.sh` rồi *Add Input* lại bản mới.


In [ ]:
import os, sys, shutil, zipfile, subprocess
from pathlib import Path

# Không clone GitHub nữa: repo private, fine-grained PAT + Kaggle Secrets tốn hai vòng
# debug mà vẫn 404. Codebase đi vào bằng Kaggle Dataset — không auth, chạy được cả khi
# Internet OFF. Đẩy bản mới bằng scripts/push_dataset.sh trên máy dev.
SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if cand.name == "legalqa" and (cand / "bm25.py").exists():
        SRC = cand.parent.parent          # .../src/legalqa -> gốc repo
        break
if SRC is None:
    # Kaggle đôi khi giữ nguyên .zip thay vì bung; tự bung vào working.
    z = next(Path("/kaggle/input").glob("**/src.zip"), None)
    assert z, "Không thấy codebase trong /kaggle/input — chưa gắn dataset legalqa-v10-src"
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/unz")
    SRC = Path("/kaggle/working/unz")

# Copy sang working: scripts ghi bm25.pkl/chunks.jsonl, /kaggle/input là read-only.
REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)

!pip -q install transformers==4.44.2 sentencepiece

os.chdir(REPO)
sys.path.insert(0, "src")

import torch
print("codebase:", (REPO / "COMMIT.txt").read_text().strip() if (REPO / "COMMIT.txt").exists() else "KHÔNG có COMMIT.txt")
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — bật Accelerator")
print("số GPU  :", torch.cuda.device_count())


## 2 — Tìm dữ liệu thi

Dò thay vì bắt gõ tay đường dẫn: tên dataset khác nhau tuỳ bạn gắn competition hay upload zip.
Mốc nhận dạng là thư mục `selected-contexts/` — thứ duy nhất chắc chắn có.

In [ ]:
from pathlib import Path

cands = [p.parent for p in Path("/kaggle/input").glob("**/selected-contexts") if p.is_dir()]
assert cands, "Không thấy selected-contexts/ trong /kaggle/input — chưa gắn dữ liệu thi"
DATA_DIR = cands[0]

train = DATA_DIR / "train.json"
assert train.exists(), f"Thiếu train.json cạnh selected-contexts/ trong {DATA_DIR}"

os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["TRAIN_JSON"] = str(train)
os.environ["CLEAN_DIR"] = "/kaggle/working/clean"
os.environ["PYTHONPATH"] = "src"

n = len(list((DATA_DIR / "selected-contexts").glob("*.json")))
print(f"DATA_DIR  {DATA_DIR}")
print(f"corpus    {n:,} file  (máy dev: 8.532)")

## 3 — clean → chunk → gold (CPU, ~12 phút)

Ba số phải khớp máy dev, lệch là dữ liệu khác bản — mọi số sau đó vô nghĩa:
`Document 8.512` · `Chunk 287.358` · `gold 6.250/7.000 = 89,3%`

In [ ]:
!mkdir -p $CLEAN_DIR
!python scripts/clean_data.py
!python scripts/build_chunks.py
!python scripts/build_gold.py
# warmup.json: 500 câu CÓ Reference answer và KHÔNG nằm trong train. Mọi ngưỡng pool/K
# đang chốt trên train -> cần tập độc lập kiểm lại, nếu không là chốt tham số trên chính
# tập đã dùng để quan sát. Máy dev dựng được 456/500 = 91,2%.
from pathlib import Path
warm = Path(os.environ["DATA_DIR"]) / "warmup.json"
if warm.exists():
    !TRAIN_JSON=$warm GOLD_OUT=gold_warmup.jsonl python scripts/build_gold.py
else:
    print("KHÔNG có warmup.json — bỏ qua tập độc lập, số chỉ chốt được trên train")


## 4 — Lấy reranker

`AITeamVN/Vietnamese_Reranker` — XLM-R large, 567.755.777 params, `XLMRobertaForSequenceClassification`,
head hồi quy có sẵn nên **không train gì cả**. Với Qwen 3B là 3,66B, còn trống 340M dưới trần 4B.

Ưu tiên bản đã gắn sẵn trong `/kaggle/input` (chạy được cả khi Internet OFF); không có thì tải HF.

In [ ]:
import json

local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "XLMRobertaForSequenceClassification" in p.read_text()]
if local:
    MODEL = str(local[0])
    print("dùng bản đã gắn:", MODEL)
else:
    from huggingface_hub import snapshot_download
    MODEL = snapshot_download("AITeamVN/Vietnamese_Reranker")
    print("tải từ HuggingFace:", MODEL)

cfg = json.loads((Path(MODEL) / "config.json").read_text())
print(cfg["architectures"], "layers", cfg["num_hidden_layers"], "hidden", cfg["hidden_size"])

## 5 — Đo

Mốc phải đấu (BM25 thuần, máy dev, n=1.000): **R@1 54,3% · R@5 81,2% · R@10 88,4% · R@50 96,7%**

Dòng đáng nhìn nhất không phải R@5 tuyệt đối mà *rerank ăn được bao nhiêu phần khoảng trống*:
rerank chỉ sắp lại pool, trần của nó là tỉ lệ gold nằm trong pool.

In [ ]:
!python scripts/eval_rerank.py --model $MODEL --pool 50 --n-cau 500 --batch-size 32

### 5b — Kiểm lại trên tập độc lập (warmup)

Cùng pool, cùng model, khác tập câu. Mốc BM25 thuần trên warmup (máy dev, n=456):
**R@1 54,2% · R@5 83,1% · R@10 89,5%** — lệch ≤1,9 điểm so với train.

Dòng `chênh` ở đây phải cùng dấu và cùng cỡ với cell 5. Lệch nhiều = số ở cell 5 là
đặc tính của tập train chứ không phải của reranker.

In [ ]:
!python scripts/eval_rerank.py --model $MODEL --pool 50 --n-cau 456 --batch-size 32 --gold gold_warmup.jsonl

## 6 — Quét pool (chạy khi cell 5 cho kết quả tốt)

Pool lớn = trần cao nhưng chậm hơn tuyến tính. Bài thi 1.000 câu, cần biết điểm gãy.
Đọc cột `s/câu` của từng lần rồi nhân 1.000.

In [ ]:
for pool in (20, 100):
    print(f"\n{'='*20} pool {pool} {'='*20}", flush=True)
    !python scripts/eval_rerank.py --model $MODEL --pool {pool} --n-cau 300 --batch-size 32